# Lab 2 · RP3beta

The runnable companion of the lab page, `docs/labs/02-rp3beta.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "rp3beta"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
item = int(data.item_pop.argmax())
for beta in (0.0, 1.0):
    m = lab.fit(METHOD, data, **{**best, "rp3_beta": beta})
    top = lab.neighbours(m, data, item)
    print(f"beta={beta}: mean popularity of the neighbours {top['popularity'].mean():.0f}")
top

**Exercise.** Reproduce the numerical trap from Level 3.2 of the lab page: build a 3 x 3 matrix where one user's weights are `np.float32(2.0 ** -140)` and call `rp3beta_similarity` on it. What do you get?

<details><summary>Hint</summary>

`from recbench.methods.neighbourhood import rp3beta_similarity`, then `rp3beta_similarity(x, 1.0, 0.5, 10).toarray()`.

</details>

<details><summary>Solution</summary>

```python
import numpy as np
import scipy.sparse as sp
from recbench.methods.neighbourhood import rp3beta_similarity

old = np.float32(2.0 ** -140)
x = sp.csr_matrix(np.array([[1.0, 1.0, 0.0], [old, 0.0, old], [0.0, 1.0, 1.0]], dtype=np.float32))
rp3beta_similarity(x, 1.0, 0.5, 10).toarray()
```
Items 0 and 2 get an `inf` weight: `1 / 1e-42` overflows float32. The lab page shows the fix.

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "rp3_beta", [0.0, 0.2, 0.4, 0.6, 0.8, 1.0], start="best")
lab.plot_sweep(table)
table

**Your prediction for `rp3_beta`, and what you saw:** ...

In [ ]:
table = lab.sweep(METHOD, DATASET, "rp3_alpha", [0.5, 1.0, 1.5, 2.0], start="best")
lab.plot_sweep(table)
table

**Your prediction for `rp3_alpha`, and what you saw:** ...

In [ ]:
table = lab.sweep(METHOD, DATASET, "decay_half_life_days", [None, 30, 90, 365], start="best")
lab.plot_sweep(table)
table

**Your prediction for `decay_half_life_days`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh rp3beta:no-popularity-penalty
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `no-popularity-penalty` is ready to run; `counts`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_rp3beta_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:no-popularity-penalty")

In [ ]:
lab.compare(METHOD, f"{METHOD}:counts")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`normalised` in `labs/02-rp3beta/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:normalised")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:normalised", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.